# РАГ — собираем и оцениваем

**Цель:** разобрать полный цикл RAG: подготовка данных, индексирование, retrieval, генерация ответа и оценка метриками.

**Используемые примеры:**
- `topic_1_rag/example_1_chunking_rag` — подготовка данных (RTF/PDF, чанкинг), распределение длин чанков
- `topic_1_rag/example_2_benchmark_rag` — бенчмарк retrieval (recall@k, MRR)
- `topic_1_rag/example_3_custom_rag` — RAG без фреймворков, оценка ROUGE/BLEU/BERTScore
- `topic_1_rag/example_4_video_rag` — мультимодальный RAG по видео (Whisper, timecode)
- `topic_1_rag/example_5_langchain_rag` — RAG на LangChain (LCEL, FAISS)
- `topic_1_rag/example_6_comparison_rag` — сравнение custom vs LangChain RAG на mirage-bench

In [ ]:
%autosave 60

In [ ]:
# pip install -r '../requirements.txt'

## Настройка окружения

Добавляем корень репозитория в `sys.path`, чтобы работали `from src.common import ...` и скрипты примеров при запуске через `%run`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
%cd ..

## Теория: пайплайн RAG

**RAG (Retrieval-Augmented Generation)** — подход, при котором перед генерацией ответа модель получает релевантные фрагменты из корпуса (базы знаний).

1. **Подготовка данных:** исходные документы разбиваются на чанки (по границам абзацев, пунктов или фиксированного размера).
2. **Индексирование:** чанки эмбеддятся (модель эмбеддингов даёт вектор на каждый чанк), векторы сохраняются в индекс (например FAISS, Annoy).
3. **Retrieval:** по запросу пользователя считают эмбеддинг запроса и находят top-k ближайших чанков (косинусное сходство или L2).
4. **Генерация:** в промпт к LLM подставляют контекст (найденные чанки) и вопрос; LLM генерирует ответ.
5. **Оценка:** качество retrieval оценивают recall@k, MRR; качество ответа — ROUGE, BLEU, BERTScore относительно эталона.

### 1. Подготовка данных

Загружаем RTF, разбиваем по нумерованным пунктам (regex), строим распределение длин чанков.

In [ ]:
%run ai_agents_course/topic_1_rag/example_1_chunking_rag/prepare_data.py

### 2. Бенчмарк retrieval

Пары (вопрос, эталон) по чанкам; метрики recall@k и MRR показывают, насколько хорошо retrieval находит нужный фрагмент.

In [ ]:
%run ai_agents_course/topic_1_rag/example_2_benchmark_rag/run_benchmark_20.py

### 3. RAG без фреймворков + оценка

Эмбеддинги чанков и запросов, top-k по косинусу, промпт с контекстом в LLM; оценка ответов через ROUGE, BLEU, BERTScore.

In [ ]:
%run ai_agents_course/topic_1_rag/example_3_custom_rag/run_rag_eval.py

### 4. Мультимодальный RAG по видео

Метаданные видео: транскрипт (Whisper) и/или описание кадров. Поиск по текстовому запросу возвращает timecode. Сначала строим метаданные (если ещё не построены), затем демо поиска.

In [ ]:
# Сбор метаданных (видео → аудио → Whisper + при необходимости кадры). Требует data/video/video.mp4 и ffmpeg.
# --force

In [ ]:
%run ai_agents_course/topic_1_rag/example_4_video_rag/build_metadata.py

In [ ]:
# Поиск по запросу (после build_metadata)

In [ ]:
%run ai_agents_course/topic_1_rag/example_4_video_rag/demo_search.py "Прогнозирование оттока клиентов"

### 5. RAG на LangChain

LCEL: RTF → чанки → FAISS, retriever + prompt + ChatOllama; один пайплайн через цепочку runnables.

In [ ]:
%run ai_agents_course/topic_1_rag/example_5_langchain_rag/run_rag_langchain.py

### 6. Сравнение custom и LangChain RAG

Один и тот же бенчмарк (mirage-bench/ru): custom RAG и RAG на LangChain; сравнение метрик в артефактах.

In [ ]:
%run ai_agents_course/topic_1_rag/example_6_comparison_rag/run_rag_eval.py

## Итоги

- Подготовка данных и чанкинг задают границы единиц поиска; от них зависят recall и релевантность контекста.
- Retrieval оценивают по recall@k и MRR; качество ответа — по ROUGE, BLEU, BERTScore.
- RAG можно собирать без фреймворков (эмбеддинги + top-k + промпт) или на LangChain (LCEL, FAISS, retriever).
- Мультимодальный RAG по видео опирается на метаданные (транскрипт, описания кадров) и поиск по ним с выдачей timecode.